# RMIT RAG Evaluation — Retrieval + DeepEval + Readability + Robustness (Out of KB and Abstentions)

This notebook covers:

- **Retrieval:** Hit@3, Recall@3, NDCG@3
- **DeepEval:** Answer Relevancy + Faithfulness
- **Readability:** Flesch Reading Ease + Flesch-Kincaid Grade Level + Gunning Fog Index + SMOG Index + answer complexity measures
- DeepEval sample: **4 per persona = 12 questions in total**


## Evaluation Approach

To evaluate the RAG system, we assess both **retrieval quality** and **generated answer quality**. This is important because a RAG system can fail at different stages: it may retrieve the wrong evidence, or it may retrieve the correct evidence but generate an inaccurate or unsupported answer.

### Retrieval Metrics

Three complementary retrieval metrics are used:

- **Hit@3** measures whether at least one relevant passage appears within the top three retrieved results. This provides a simple indication of whether the retriever is able to surface useful evidence for a query.

- **Recall@3** measures the proportion of all relevant passages that are retrieved within the top three results. This is useful for identifying cases where the retriever finds some relevant evidence but misses other important supporting passages.

- **NDCG@3** evaluates both relevance and ranking position. Relevant passages receive greater credit when they appear higher in the retrieved results, making this metric useful for assessing whether the most useful evidence is prioritised.

These metrics were chosen together because they capture different aspects of retrieval performance: **Hit@3 measures presence, Recall@3 measures coverage, and NDCG@3 measures ranking quality**.

### Generation Metrics

For answer generation, **DeepEval** is used to evaluate:

- **Answer Relevancy** — whether the generated response directly addresses the user's question.
- **Faithfulness** — whether the generated response is supported by the retrieved context rather than introducing unsupported information.

DeepEval was selected because these metrics align closely with the main goals of a RAG system: responses should be both **relevant to the user's query** and **grounded in the retrieved source material**.

Using DeepEval alongside retrieval metrics allows the system to be evaluated as a complete RAG pipeline rather than assessing retrieval or generation in isolation. This distinction is important because strong retrieval performance does not necessarily guarantee a correct final answer, and a well-written answer may still be unreliable if it is not supported by the retrieved evidence.

### Answer Complexity (Readability) Metrics

For generated answers, readability metrics are used to evaluate the complexity and accessibility of responses for a reader:

* **Flesch Reading Ease** — estimates how easy the response is to read. Higher scores indicate easier-to-read text.
* **Flesch-Kincaid Grade Level** — estimates the US school grade level required to understand the response. Lower scores indicate simpler text.
* **Gunning Fog Index** — estimates the number of years of formal education generally needed to understand the response on a first reading. Lower scores indicate simpler text and fewer complex words.
* **SMOG Index** — estimates the education level needed to understand a response based on the frequency of complex, multi-syllable words. Lower scores indicate simpler text. As chatbot answers are often relatively short, SMOG scores should be interpreted cautiously.
* **Average words per sentence** — measures sentence length and provides an additional indicator of answer complexity. Longer sentences may indicate more complex sentence structures.
* **Average syllables per word** — measures word complexity at a basic lexical level. Higher values generally indicate the use of longer or more complex words.

These metrics complement **Answer Relevancy** and **Faithfulness** by evaluating the **presentation, readability, and accessibility** of generated answers rather than whether the answers are relevant or factually supported. They are descriptive measures and should not be interpreted as definitive assessments of answer quality or suitability for a particular reader.


In [5]:
import os
import re
import time
import numpy as np
import pandas as pd
import ollama

from rank_bm25 import BM25Okapi

pd.set_option("display.max_colwidth", 140)

TOPICS_FILE = "topics_WIL20.csv"
PASSAGES_FILE = "passages_WIL20.csv"
QRELS_FILE = "qrels_WIL20.txt"

GENERATOR_MODEL = "llama3.2:3b"
JUDGE_MODEL = "qwen3:1.7b"

TOP_K = 3
DEEPEVAL_PER_PERSONA = 4
RANDOM_STATE = 42

os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "180"
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "600"
os.environ["DEEPEVAL_RETRY_MAX_ATTEMPTS"] = "1"


## Load data

In [6]:
topics = pd.read_csv(TOPICS_FILE)
passages_df = pd.read_csv(PASSAGES_FILE)

qrels = pd.read_csv(
    QRELS_FILE,
    sep=r"\s+",
    header=None,
    names=["question_id", "unused", "passage_id", "relevance"]
)

print("Topics:", topics.shape)
print("Passages:", passages_df.shape)
print("Qrels:", qrels.shape)

display(topics.head())
display(passages_df.head())
display(qrels.head())


Topics: (70, 6)
Passages: (45, 4)
Qrels: (73, 4)


,topic_id,topic,question_id,question,persona,status
0,C01,Can I study a Bachelor of Business online?,C01Q01,Can I study a Bachelor of Business online?,prospective student,known
1,C01,Can I study a Bachelor of Business online?,C01Q02,Is there an online version of the Business degree?,current student,known
2,C02,What career outcomes does the Marketing major lead to?,C02Q01,What career outcomes does the Marketing major lead to?,parent/guardian,known
3,C02,What career outcomes does the Marketing major lead to?,C02Q02,What jobs can students expect after majoring in Marketing?,current student,known
4,C03,Can I enrol in advanced electives early?,C03Q01,Am I eligible to take a third-year elective as a first-year student?,current student,known


,passage_id,passage,school,program
0,P01,The Bachelor of Business is available to study online at RMIT. The online Bachelor of Business takes 36 months to complete and offers fl...,"Economics, Finance and Marketing",Bachelor of Business
1,P02,"The Marketing major prepares graduates for roles in digital marketing, brand management, campaign strategy, and customer analytics acros...","Economics, Finance and Marketing",Bachelor of Business
2,P03,"At RMIT, you can take a third-year elective as a first-year student as long as you meet the course requirements and your program structu...","Economics, Finance and Marketing",General
3,P04,"When you successfully complete this degree, you may be eligible for entry into a range of RMIT Honours and postgraduate qualifications i...","Economics, Finance and Marketing",Bachelor of Commerce
4,P05,"The Bachelor of Graphic Design offers four majors: Branding, Experience Design, Illustration, and Typography. The Branding major focuses...",Design,Bachelor of Graphic Design


,question_id,unused,passage_id,relevance
0,C01Q01,0,P01,2
1,C01Q01,0,P15,1
2,C01Q02,0,P01,2
3,C01Q02,0,P15,1
4,C02Q01,0,P02,2


# Retrieval

In [7]:
def tokenize(text):
    return re.findall(r"\b\w+\b", str(text).lower())

tokenised_passages = [
    tokenize(text)
    for text in passages_df["passage"].fillna("").tolist()
]

bm25 = BM25Okapi(tokenised_passages)

def retrieve_top_k(question, k=TOP_K):
    scores = bm25.get_scores(tokenize(question))
    top_indices = np.argsort(scores)[::-1][:k]

    return [
        {
            "rank": rank,
            "passage_id": passages_df.iloc[i]["passage_id"],
            "passage": passages_df.iloc[i]["passage"],
            "score": float(scores[i]),
        }
        for rank, i in enumerate(top_indices, start=1)
    ]


In [8]:
qrels_by_question = {}

for question_id, group in qrels.groupby("question_id"):
    qrels_by_question[question_id] = dict(
        zip(group["passage_id"], group["relevance"])
    )

def dcg(relevances):
    return sum(
        (2 ** rel - 1) / np.log2(rank + 2)
        for rank, rel in enumerate(relevances)
    )

def retrieval_metrics_for_question(question_id, question, k=TOP_K):
    judged = qrels_by_question.get(question_id)

    if not judged:
        return None

    retrieved = retrieve_top_k(question, k=k)
    retrieved_ids = [x["passage_id"] for x in retrieved]

    relevant_ids = {
        pid for pid, rel in judged.items()
        if rel > 0
    }

    hit = int(any(pid in relevant_ids for pid in retrieved_ids))

    recall = (
        len(set(retrieved_ids) & relevant_ids) / len(relevant_ids)
        if relevant_ids else np.nan
    )

    retrieved_rels = [
        int(judged.get(pid, 0))
        for pid in retrieved_ids
    ]

    ideal_rels = sorted(
        [int(rel) for rel in judged.values()],
        reverse=True
    )[:k]

    retrieved_rels += [0] * (k - len(retrieved_rels))
    ideal_rels += [0] * (k - len(ideal_rels))

    ideal_dcg = dcg(ideal_rels)

    ndcg = dcg(retrieved_rels) / ideal_dcg if ideal_dcg > 0 else np.nan

    return {
        "retrieved_passage_ids": retrieved_ids,
        f"hit@{k}": hit,
        f"recall@{k}": recall,
        f"ndcg@{k}": ndcg,
    }


In [9]:
retrieval_rows = []

for _, row in topics.iterrows():
    metrics = retrieval_metrics_for_question(
        row["question_id"],
        row["question"],
        k=TOP_K
    )

    if metrics is not None:
        retrieval_rows.append({
            "question_id": row["question_id"],
            "persona": row["persona"],
            "question": row["question"],
            **metrics
        })

retrieval_results = pd.DataFrame(retrieval_rows)

display(retrieval_results.head())
print("Evaluated questions:", len(retrieval_results))


,question_id,persona,question,retrieved_passage_ids,hit@3,recall@3,ndcg@3
0,C01Q01,prospective student,Can I study a Bachelor of Business online?,"[P01, P16, P15]",1,1.0,0.963940
1,C01Q02,current student,Is there an online version of the Business degree?,"[P01, P16, P44]",1,0.5,0.826235
2,C02Q01,parent/guardian,What career outcomes does the Marketing major lead to?,"[P02, P32, P14]",1,1.0,1.000000
3,C02Q02,current student,What jobs can students expect after majoring in Marketing?,"[P02, P04, P44]",1,1.0,1.000000
4,C03Q01,current student,Am I eligible to take a third-year elective as a first-year student?,"[P03, P19, P12]",1,1.0,1.000000


Evaluated questions: 53


In [10]:
retrieval_summary = pd.DataFrame({
    "metric": [f"Hit@{TOP_K}", f"Recall@{TOP_K}", f"NDCG@{TOP_K}"],
    "score": [
        retrieval_results[f"hit@{TOP_K}"].mean(),
        retrieval_results[f"recall@{TOP_K}"].mean(),
        retrieval_results[f"ndcg@{TOP_K}"].mean(),
    ]
})

display(retrieval_summary.round(3))


,metric,score
0,Hit@3,0.774
1,Recall@3,0.717
2,NDCG@3,0.686


In [11]:
retrieval_by_persona = (
    retrieval_results
    .groupby("persona")[
        [f"hit@{TOP_K}", f"recall@{TOP_K}", f"ndcg@{TOP_K}"]
    ]
    .mean()
    .round(3)
)

display(retrieval_by_persona)


,hit@3,recall@3,ndcg@3
persona,,,
current student,0.947,0.868,0.850
parent/guardian,0.692,0.654,0.599
prospective student,0.667,0.619,0.591


In [12]:
hit_failures = retrieval_results[
    retrieval_results[f"hit@{TOP_K}"] == 0
].copy()

print("Hit@3 failures:", len(hit_failures))

display(
    hit_failures[
        ["question_id", "persona", "question", "retrieved_passage_ids"]
    ]
)


Hit@3 failures: 12


,question_id,persona,question,retrieved_passage_ids
6,C04Q01,parent/guardian,Does the Bachelor of Commerce prepare students for further postgraduate study?,"[P44, P20, P38]"
8,C05Q01,prospective student,What majors can I choose from in the Bachelor of Graphic Design?,"[P40, P06, P41]"
9,C05Q02,prospective student,What courses does Bachelor of Graphic Design offer?,"[P15, P25, P40]"
10,C06Q01,prospective student,What career opportunities are available after completing the Bachelor of Graphic Design?,"[P44, P23, P40]"
12,C07Q01,prospective student,What does the Bachelor of Games focus on?,"[P24, P32, P39]"
13,C07Q02,current student,What areas will I study in the Bachelor of Games?,"[P20, P35, P43]"
15,C08Q02,prospective student,What English requirements do I need to meet for the Bachelor of Games?,"[P31, P18, P43]"
23,C14Q01,prospective student,What software should I be expected to learn during the Bachelor of Games?,"[P16, P17, P44]"
24,C14Q02,prospective student,What software do I need to have in order to complete the Bachelor of games?,"[P11, P18, P32]"
33,C23Q01,parent/guardian,Will students get any industry experience while completing Commerce?,"[P33, P25, P20]"


In [13]:
retrieval_results.to_csv("retrieval_evaluation_results.csv", index=False)
retrieval_summary.to_csv("retrieval_evaluation_summary.csv", index=False)
retrieval_by_persona.to_csv("retrieval_evaluation_by_persona.csv")

print("Retrieval results saved.")


Retrieval results saved.


# Generate answers for DeepEval

In [14]:
ABSTENTION_TEXT = (
    "I don't have enough information in the provided RMIT sources "
    "to answer this question."
)

def build_prompt(question, context_passages):
    context = "\n\n".join(context_passages)

    return f"""You are an RMIT university information assistant.

Answer the user's question using ONLY the RMIT information provided below.

Follow these rules carefully:

1. Read ALL provided passages before answering.

2. If a passage directly answers the question, use that information.
   Do NOT abstain when the answer is explicitly stated.

3. Prioritise the passage that most specifically matches the user's question.
   For example:
   - for part-time study, prioritise information specifically about part-time study;
   - for changing majors, prioritise information specifically about changing majors;
   - for a named program, only use information that applies to that program.

4. Preserve explicit statements exactly.
   Pay particular attention to:
   "can", "cannot", "must", "must not",
   "required", "not required",
   "eligible", "not eligible",
   "exempt", "not exempt",
   "full-time", and "part-time".

   Also preserve numerical information such as years, subjects, fees,
   scores, study loads, and durations.

5. Do not infer permission, eligibility, requirements, policies, predictions,
   comparisons, or outcomes from indirect or missing information.

   The absence of information does NOT mean the answer is "no".

6. ANSWER IN A COMPLETE SENTENCE
   Give a direct, self-contained answer to the user's question.

   Do not respond with only "Yes" or "No".

   For yes-or-no questions, clearly state whether the answer is yes or no,
   then include the specific information from the RMIT sources that supports it.

   For questions beginning with "what", "how", "which", "when", or similar
   question words, answer the requested information directly.
   Do not begin these answers with "Yes" or "No".

   Make sure the answer does not contradict the evidence.

7. Before responding, check that your answer does not contradict any explicit
   statement in the provided information.

8. Only abstain if none of the provided passages contain enough information
   to answer the question.

   If there is not enough information, respond exactly with:
   "{ABSTENTION_TEXT}"

9. ANSWER DIRECTLY AND CONCISELY
   Answer in one or two complete sentences where possible.

   Include enough information to fully answer the question, but do not add
   unnecessary details.

   Do not explain your reasoning process or describe how you found the answer.
   Do not use phrases such as:
   "Based on the information provided",
   "According to the provided information",
   or "To determine this".

10. Do not invent, assume, predict, calculate, or add information that is not
    supported by the RMIT information below.

Question:
{question}

RMIT information:
{context}

Answer:
"""

def generate_answer(question, model=GENERATOR_MODEL, k=TOP_K):
    retrieved = retrieve_top_k(question, k=k)
    context_passages = [item["passage"] for item in retrieved]

    response = ollama.chat(
        model=model,
        messages=[{
            "role": "user",
            "content": build_prompt(question, context_passages)
        }],
        options={
            "temperature": 0.0,
            "num_ctx": 4096
        }
    )

    return {
        "answer": response["message"]["content"].strip(),
        "retrieval_context": context_passages,
        "retrieved_passage_ids": [item["passage_id"] for item in retrieved]
    }


## Build a small stratified sample

In [15]:
qrel_question_ids = set(qrels["question_id"])

deepeval_pool = topics[
    (topics["status"] == "known") &
    (topics["question_id"].isin(qrel_question_ids))
].copy()

print("Eligible questions per persona:")
display(deepeval_pool["persona"].value_counts())

deepeval_sample = (
    deepeval_pool
    .groupby("persona", group_keys=False)
    .sample(
        n=DEEPEVAL_PER_PERSONA,
        random_state=RANDOM_STATE
    )
    .reset_index(drop=True)
)

display(
    deepeval_sample[
        ["question_id", "persona", "question"]
    ].sort_values(["persona", "question_id"])
)

print("DeepEval sample size:", len(deepeval_sample))


Eligible questions per persona:


prospective student    20
current student        19
parent/guardian        13
Name: persona, dtype: int64

,question_id,persona,question
0,C01Q02,current student,Is there an online version of the Business degree?
3,C02Q02,current student,What jobs can students expect after majoring in Marketing?
1,C06Q02,current student,What jobs can I pursue after graduating from the Bachelor of Graphic Design?
2,C20Q01,current student,How can I structure my degree part-time if I plan on working part time?
7,C17Q02,parent/guardian,Can a student switch majors halfway through the Bachelor of Laws + Commerce program?
6,C24Q01,parent/guardian,How much does the Bachelor of Business cost?
5,C25Q01,parent/guardian,What computer specifications will my child need for the Bachelor of games?
4,C51Q01,parent/guardian,Will my child have opportunities to work on real-world design projects?
10,C05Q01,prospective student,What majors can I choose from in the Bachelor of Graphic Design?
8,C08Q01,prospective student,What are the prerequisites for the Bachelor of Games?


DeepEval sample size: 12


## Generate answers first

In [16]:
generation_rows = []

for i, row in deepeval_sample.iterrows():
    print(
        f"[{i + 1}/{len(deepeval_sample)}] "
        f"Generating {row['question_id']}..."
    )

    result = generate_answer(row["question"])

    generation_rows.append({
        "question_id": row["question_id"],
        "persona": row["persona"],
        "question": row["question"],
        "answer": result["answer"],
        "retrieved_passage_ids": result["retrieved_passage_ids"],
        "retrieval_context": result["retrieval_context"]
    })

    pd.DataFrame(generation_rows).to_pickle(
        "generation_results_progress.pkl"
    )

generation_results = pd.DataFrame(generation_rows)

display(
    generation_results[
        ["question_id", "persona", "question", "answer", "retrieved_passage_ids"]
    ]
)

print("Generation complete.")


[1/12] Generating C01Q02...
[2/12] Generating C06Q02...
[3/12] Generating C20Q01...
[4/12] Generating C02Q02...
[5/12] Generating C51Q01...
[6/12] Generating C25Q01...
[7/12] Generating C24Q01...
[8/12] Generating C17Q02...
[9/12] Generating C08Q01...
[10/12] Generating C42Q01...
[11/12] Generating C05Q01...
[12/12] Generating C09Q01...


,question_id,persona,question,answer,retrieved_passage_ids
0,C01Q02,current student,Is there an online version of the Business degree?,"Yes, there is an online version of the Business degree, specifically the Bachelor of Business, which is available to study online at RMIT.","[P01, P16, P44]"
1,C06Q02,current student,What jobs can I pursue after graduating from the Bachelor of Graphic Design?,"Graduates of the Bachelor of Graphic Design can pursue careers including graphic designer, UX/UI designer, brand designer, illustrator, ...","[P41, P06, P14]"
2,C20Q01,current student,How can I structure my degree part-time if I plan on working part time?,"To structure your degree part-time while working part-time, you can complete four courses per year over six years, as part-time students...","[P15, P27, P35]"
3,C02Q02,current student,What jobs can students expect after majoring in Marketing?,"Students majoring in Marketing can expect roles in digital marketing, brand management, campaign strategy, and customer analytics across...","[P02, P04, P44]"
4,C51Q01,parent/guardian,Will my child have opportunities to work on real-world design projects?,"Your child will have opportunities to work on real-world design projects through the Bachelor of Graphic Design, which includes industry...","[P24, P37, P36]"
5,C25Q01,parent/guardian,What computer specifications will my child need for the Bachelor of games?,I don't have enough information in the provided RMIT sources to answer this question.,"[P18, P33, P07]"
6,C24Q01,parent/guardian,How much does the Bachelor of Business cost?,The cost of the Bachelor of Business is not specified in the provided information.,"[P14, P32, P44]"
7,C17Q02,parent/guardian,Can a student switch majors halfway through the Bachelor of Laws + Commerce program?,"A student cannot switch majors halfway through the Bachelor of Laws + Commerce program. This is because, once a student has commenced th...","[P26, P27, P25]"
8,C08Q01,prospective student,What are the prerequisites for the Bachelor of Games?,"To be eligible for the Bachelor of Games, applicants must complete a selection task, which is a requirement for the program.","[P18, P03, P17]"
9,C42Q01,prospective student,How long does the online Bachelor of Business take if I study part-time?,The online Bachelor of Business takes 36 months to complete if you study part-time.,"[P15, P03, P01]"


Generation complete.


## Readability Evaluation of Generated Answers

Readability is evaluated on the generated answers from the 12-question stratified sample. The metrics describe the linguistic complexity of each response and can be compared across personas.


In [17]:
def count_syllables(word):
    """Approximate the number of syllables in an English word."""
    word = re.sub(r"[^a-zA-Z]", "", str(word)).lower()

    if not word:
        return 0

    # Treat a final silent 'e' as non-syllabic in most cases.
    word = re.sub(r"e$", "", word)
    vowel_groups = re.findall(r"[aeiouy]+", word)
    syllables = len(vowel_groups)

    return max(1, syllables)


def readability_metrics(text):
    """Calculate descriptive readability metrics for a generated answer."""
    text = str(text).strip()
    words = re.findall(r"\b[A-Za-z]+(?:['-][A-Za-z]+)*\b", text)
    sentences = [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]

    word_count = len(words)
    sentence_count = len(sentences)
    syllable_count = sum(count_syllables(word) for word in words)

    if word_count == 0 or sentence_count == 0:
        return {
            "word_count": 0,
            "sentence_count": 0,
            "complex_word_count": 0,
            "avg_words_per_sentence": np.nan,
            "avg_syllables_per_word": np.nan,
            "flesch_reading_ease": np.nan,
            "flesch_kincaid_grade": np.nan,
            "gunning_fog": np.nan,
            "smog_index": np.nan,
        }

    words_per_sentence = word_count / sentence_count
    syllables_per_word = syllable_count / word_count

    # Complex words are approximated as words containing 3+ syllables.
    complex_word_count = sum(count_syllables(word) >= 3 for word in words)

    # Standard Flesch formulas.
    flesch_reading_ease = (
        206.835
        - 1.015 * words_per_sentence
        - 84.6 * syllables_per_word
    )

    flesch_kincaid_grade = (
        0.39 * words_per_sentence
        + 11.8 * syllables_per_word
        - 15.59
    )

    # Gunning Fog Index.
    gunning_fog = 0.4 * (
        words_per_sentence
        + 100 * (complex_word_count / word_count)
    )

    # SMOG Index.
    # The standard formula is most appropriate for longer passages;
    # short chatbot answers should therefore be interpreted cautiously.
    smog_index = (
        1.0430 * np.sqrt(complex_word_count * (30 / sentence_count))
        + 3.1291
    )

    return {
        "word_count": word_count,
        "sentence_count": sentence_count,
        "complex_word_count": complex_word_count,
        "avg_words_per_sentence": words_per_sentence,
        "avg_syllables_per_word": syllables_per_word,
        "flesch_reading_ease": flesch_reading_ease,
        "flesch_kincaid_grade": flesch_kincaid_grade,
        "gunning_fog": gunning_fog,
        "smog_index": smog_index,
    }


In [18]:
readability_rows = []

for _, row in generation_results.iterrows():
    metrics = readability_metrics(row["answer"])

    readability_rows.append({
        "question_id": row["question_id"],
        "persona": row["persona"],
        **metrics
    })

readability_results = pd.DataFrame(readability_rows)

display(
    readability_results[
        [
            "question_id",
            "persona",
            "word_count",
            "sentence_count",
            "avg_words_per_sentence",
            "avg_syllables_per_word",
            "complex_word_count",
            "flesch_reading_ease",
            "flesch_kincaid_grade",
            "gunning_fog",
            "smog_index"
        ]
    ].round(2)
)


,question_id,persona,word_count,sentence_count,avg_words_per_sentence,avg_syllables_per_word,complex_word_count,flesch_reading_ease,flesch_kincaid_grade,gunning_fog,smog_index
0,C01Q02,current student,23,1,23.0,1.74,5,36.36,13.90,17.90,15.90
1,C06Q02,current student,31,1,31.0,2.10,11,-2.02,21.24,26.59,22.08
2,C20Q01,current student,27,1,27.0,1.56,2,47.83,13.30,13.76,11.21
3,C02Q02,current student,42,1,42.0,2.14,16,-17.08,26.08,32.04,25.98
4,C51Q01,parent/guardian,33,1,33.0,1.85,6,16.96,19.09,20.47,17.12
5,C25Q01,parent/guardian,14,1,14.0,1.64,2,53.64,9.26,11.31,11.21
6,C24Q01,parent/guardian,14,1,14.0,1.79,5,41.55,10.94,19.89,15.90
7,C17Q02,parent/guardian,35,2,17.5,1.60,5,53.71,10.12,12.71,12.16
8,C08Q01,prospective student,21,1,21.0,1.67,5,44.52,12.27,17.92,15.90
9,C42Q01,prospective student,13,1,13.0,1.69,2,50.47,9.45,11.35,11.21


In [19]:
readability_summary = pd.DataFrame({
    "metric": [
        "Average words per sentence",
        "Average syllables per word",
        "Average complex words per answer",
        "Flesch Reading Ease",
        "Flesch-Kincaid Grade Level",
        "Gunning Fog Index",
        "SMOG Index"
    ],
    "score": [
        readability_results["avg_words_per_sentence"].mean(),
        readability_results["avg_syllables_per_word"].mean(),
        readability_results["complex_word_count"].mean(),
        readability_results["flesch_reading_ease"].mean(),
        readability_results["flesch_kincaid_grade"].mean(),
        readability_results["gunning_fog"].mean(),
        readability_results["smog_index"].mean()
    ]
})

print("Overall readability metrics:")
display(readability_summary.round(2))


Overall readability metrics:


,metric,score
0,Average words per sentence,24.54
1,Average syllables per word,1.80
2,Average complex words per answer,6.25
3,Flesch Reading Ease,29.79
4,Flesch-Kincaid Grade Level,15.20
5,Gunning Fog Index,19.01
6,SMOG Index,16.35


In [20]:
readability_by_persona = (
    readability_results
    .groupby("persona")[
        [
            "avg_words_per_sentence",
            "avg_syllables_per_word",
            "flesch_reading_ease",
            "flesch_kincaid_grade",
            "gunning_fog",
            "smog_index"
        ]
    ]
    .mean()
    .round(2)
)

print("Readability by persona:")
display(readability_by_persona)


Readability by persona:


,avg_words_per_sentence,avg_syllables_per_word,flesch_reading_ease,flesch_kincaid_grade,gunning_fog,smog_index
persona,,,,,,
current student,30.75,1.88,16.27,18.63,22.57,18.79
parent/guardian,19.62,1.72,41.47,12.35,16.10,14.10
prospective student,23.25,1.79,31.63,14.62,18.37,16.15


In [38]:
readability_results.to_csv("readability_results.csv", index=False)
readability_summary.to_csv("readability_summary.csv", index=False)
readability_by_persona.to_csv("readability_by_persona.csv")

print("Readability results saved.")


Readability results saved.


# DeepEval

In [39]:
from deepeval.models import OllamaModel
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric
from deepeval.test_case import LLMTestCase

judge_model = OllamaModel(
    model=JUDGE_MODEL,   
    base_url="http://localhost:11434",
    temperature=0
)

answer_relevancy_metric = AnswerRelevancyMetric(
    threshold=0.5,
    model=judge_model,
    include_reason=False,
    async_mode=True
)

faithfulness_metric = FaithfulnessMetric(
    threshold=0.5,
    model=judge_model,
    include_reason=False,
    async_mode=False
)

print("DeepEval judge:", JUDGE_MODEL)

ModuleNotFoundError: No module named 'deepeval'

## Answer Relevancy

In [ ]:
relevancy_rows = []

for i, row in generation_results.iterrows():
    print(
        f"[{i + 1}/{len(generation_results)}] "
        f"Answer Relevancy — {row['question_id']}"
    )

    test_case = LLMTestCase(
        input=row["question"],
        actual_output=row["answer"]
    )

    try:
        answer_relevancy_metric.measure(test_case)
        score = answer_relevancy_metric.score
        error = None

    except Exception as e:
        score = np.nan
        error = str(e)

    relevancy_rows.append({
        "question_id": row["question_id"],
        "answer_relevancy": score,
        "relevancy_error": error
    })

    pd.DataFrame(relevancy_rows).to_csv(
        "deepeval_relevancy_progress.csv",
        index=False
    )

    print(f"Score: {score}")

relevancy_results = pd.DataFrame(relevancy_rows)

display(relevancy_results)

## Faithfulness

In [ ]:
faithfulness_rows = []

for i, row in generation_results.iterrows():
    print(
        f"[{i + 1}/{len(generation_results)}] "
        f"Faithfulness — {row['question_id']}"
    )

    test_case = LLMTestCase(
        input=row["question"],
        actual_output=row["answer"],
        retrieval_context=row["retrieval_context"]
    )

    try:
        faithfulness_metric.measure(test_case)
        score = faithfulness_metric.score
        error = None
    except Exception as e:
        score = np.nan
        error = str(e)

    faithfulness_rows.append({
        "question_id": row["question_id"],
        "faithfulness": score,
        "faithfulness_error": error
    })

    pd.DataFrame(faithfulness_rows).to_csv(
        "deepeval_faithfulness_progress.csv",
        index=False
    )

    print(f"Score: {score}")

faithfulness_results = pd.DataFrame(faithfulness_rows)
display(faithfulness_results)

In [ ]:
for qid in ["C01Q02", "C06Q02", "C02Q02", "C20Q01", "C51Q01", "C25Q01", "C24Q01", "C17Q02",
            "C08Q01", "C42Q01", "C05Q01", "C09Q01"]:
    row = generation_results[
        generation_results["question_id"] == qid
    ].iloc[0]

    print("\nQUESTION:", qid)
    print(row["question"])

    print("\nANSWER:")
    print(row["answer"])

    print("\nRETRIEVAL CONTEXT:")
    for passage in row["retrieval_context"]:
        print("-", passage)

    print("\n" + "=" * 80)

# Robustness (Out-of-KB Refusal & False Abstention)

## Categorise the out-of-KB questions

- **predictive** — asks about the future / not-yet-decided information
- **subjective** — asks for an opinion or comparison with no objective answer ("hardest degree", "best lecturer")
- **personal_circumstance** — depends on applicant-specific info the KB can never contain (chances of acceptance)
- **out_of_scope_topic** — a real, objective topic, just outside the course-structure domain (salary, software, job market)
- **kb_gap** — a genuinely in-scope, answerable-in-principle question that the current KB simply doesn't cover

In [23]:
OOK_SUBTYPES = {
    "C14Q02": "out_of_scope_topic",       # software needed for Bachelor of Games
    "C18Q01": "kb_gap",                   # transferring Business <-> Commerce
    "C27Q01": "subjective",               # Graphic Design vs Games difficulty
    "C27Q02": "subjective",
    "C28Q01": "predictive",               # ATAR in 2028
    "C28Q02": "predictive",
    "C29Q01": "subjective",               # "best" employment rate
    "C29Q02": "out_of_scope_topic",       # employability after Games
    "C30Q01": "subjective",               # "best" lecturer
    "C31Q01": "personal_circumstance",    # chances of acceptance
    "C31Q02": "personal_circumstance",
    "C32Q01": "out_of_scope_topic",       # laptop recommendation
    "C33Q01": "out_of_scope_topic",       # salary
    "C33Q02": "out_of_scope_topic",
    "C34Q01": "out_of_scope_topic",       # job openings
    "C35Q01": "predictive",               # new majors in 2 years
    "C35Q02": "predictive",
    "C52Q01": "predictive",               # job market in 5 years
}

ook_topics = topics[topics["status"] == "out-of-kb"].copy()
ook_topics["subtype"] = ook_topics["question_id"].map(OOK_SUBTYPES)

missing = ook_topics[ook_topics["subtype"].isna()]
if len(missing):
    print("WARNING: unmapped out-of-kb question_ids, add them to OOK_SUBTYPES:")
    display(missing[["question_id", "question"]])

display(ook_topics[["question_id", "persona", "subtype", "question"]])
print("Out-of-KB questions:", len(ook_topics))
print(ook_topics["subtype"].value_counts())


,question_id,persona,subtype,question
24,C14Q02,prospective student,out_of_scope_topic,What software do I need to have in order to complete the Bachelor of games?
29,C18Q01,current student,kb_gap,Can I transfer between the Bachelor of Business to the Bachelor of Commerce and vice versa?
38,C27Q01,prospective student,subjective,Is the Bachelor of Graphic Design harder than the Bachelor of Games?
39,C27Q02,prospective student,subjective,"Which is more difficult, the Bachelor of Games or Graphic Design?"
40,C28Q01,prospective student,predictive,Is the ATAR for Bachelor of Business going to change in 2028?
41,C28Q02,prospective student,predictive,What ATAR will I need for the Bachelor of Business in 2028?
42,C29Q02,current student,out_of_scope_topic,What is my employability looking like after completing a Bachelor of Games?
43,C29Q01,prospective student,subjective,Which degree has the best employment rate?
44,C30Q01,current student,subjective,Which lecturer is the best for the Bachelor of Commerce subjects?
45,C31Q01,prospective student,personal_circumstance,What are the chances my application will be accepted?


Out-of-KB questions: 18
out_of_scope_topic       6
predictive               5
subjective               4
personal_circumstance    2
kb_gap                   1
Name: subtype, dtype: int64


## Generate answers for the out-of-KB questions

In [24]:
ook_generation_rows = []

for i, row in ook_topics.iterrows():
    print(f"[{len(ook_generation_rows) + 1}/{len(ook_topics)}] Generating {row['question_id']}...")

    result = generate_answer(row["question"])

    ook_generation_rows.append({
        "question_id": row["question_id"],
        "persona": row["persona"],
        "subtype": row["subtype"],
        "question": row["question"],
        "answer": result["answer"],
        "retrieved_passage_ids": result["retrieved_passage_ids"],
        "retrieval_context": result["retrieval_context"],
    })

    pd.DataFrame(ook_generation_rows).to_pickle("ook_generation_results_progress.pkl")

ook_generation_results = pd.DataFrame(ook_generation_rows)
display(ook_generation_results[["question_id", "persona", "subtype", "question", "answer"]])
print("Out-of-KB generation complete.")


[1/18] Generating C14Q02...
[2/18] Generating C18Q01...
[3/18] Generating C27Q01...
[4/18] Generating C27Q02...
[5/18] Generating C28Q01...
[6/18] Generating C28Q02...
[7/18] Generating C29Q02...
[8/18] Generating C29Q01...
[9/18] Generating C30Q01...
[10/18] Generating C31Q01...
[11/18] Generating C31Q02...
[12/18] Generating C32Q01...
[13/18] Generating C33Q01...
[14/18] Generating C33Q02...
[15/18] Generating C34Q01...
[16/18] Generating C35Q01...
[17/18] Generating C35Q02...
[18/18] Generating C52Q01...


,question_id,persona,subtype,question,answer
0,C14Q02,prospective student,out_of_scope_topic,What software do I need to have in order to complete the Bachelor of games?,"To complete the Bachelor of Games, you do not need to have specific software, as the program is more focused on the design and developme..."
1,C18Q01,current student,kb_gap,Can I transfer between the Bachelor of Business to the Bachelor of Commerce and vice versa?,"You can transfer between the Bachelor of Business and the Bachelor of Commerce, but the transfer is not explicitly stated in the provide..."
2,C27Q01,prospective student,subjective,Is the Bachelor of Graphic Design harder than the Bachelor of Games?,I don't have enough information in the provided RMIT sources to answer this question.
3,C27Q02,prospective student,subjective,"Which is more difficult, the Bachelor of Games or Graphic Design?",I don't have enough information in the provided RMIT sources to answer this question.
4,C28Q01,prospective student,predictive,Is the ATAR for Bachelor of Business going to change in 2028?,I don't have enough information in the provided RMIT sources to answer this question.
5,C28Q02,prospective student,predictive,What ATAR will I need for the Bachelor of Business in 2028?,I don't have enough information in the provided RMIT sources to answer this question.
6,C29Q02,current student,out_of_scope_topic,What is my employability looking like after completing a Bachelor of Games?,I don't have enough information in the provided RMIT sources to answer this question.
7,C29Q01,prospective student,subjective,Which degree has the best employment rate?,I don't have enough information in the provided RMIT sources to answer this question.
8,C30Q01,current student,subjective,Which lecturer is the best for the Bachelor of Commerce subjects?,I don't have enough information in the provided RMIT sources to answer this question.
9,C31Q01,prospective student,personal_circumstance,What are the chances my application will be accepted?,I don't have enough information in the provided RMIT sources to answer this question.


Out-of-KB generation complete.


## Classify each answer as abstained or attempted

In [25]:
import difflib

def fuzzy_is_abstention(answer, threshold=0.6):
    """Cheap heuristic: does the answer closely resemble the canonical abstention text,
    or open with an equivalent refusal phrase?"""
    answer_clean = str(answer).strip().lower()
    ratio = difflib.SequenceMatcher(
        None, answer_clean, ABSTENTION_TEXT.lower()
    ).ratio()

    refusal_phrases = [
        "don't have enough information",
        "do not have enough information",
        "not enough information",
        "cannot answer",
        "can't answer",
        "unable to answer",
        "no information",
        "not provided",
        "not mentioned in the",
    ]
    phrase_hit = any(p in answer_clean for p in refusal_phrases)

    return (ratio >= threshold) or phrase_hit


JUDGE_ABSTENTION_PROMPT = """You are auditing an RMIT university chatbot.

Question asked:
{question}

Chatbot's answer:
{answer}

Decide whether the chatbot actually ANSWERED the question with specific information,
or whether it ABSTAINED / refused / said it did not have enough information
(including partial refusals, hedged non-answers, or "I'm not sure" type responses).

Respond with exactly one word: ANSWERED or ABSTAINED."""


def judge_is_abstention(question, answer, model=JUDGE_MODEL, retries=2):
    prompt = JUDGE_ABSTENTION_PROMPT.format(question=question, answer=answer)

    for attempt in range(retries):
        response = ollama.chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.0}
        )
        verdict = response["message"]["content"].strip().upper()

        if "ABSTAIN" in verdict:
            return True
        if "ANSWER" in verdict:
            return False

    return fuzzy_is_abstention(answer)


def classify_abstention_row(question, answer):
    fuzzy = fuzzy_is_abstention(answer)
    judged = judge_is_abstention(question, answer)

    return {
        "fuzzy_abstained": fuzzy,
        "judge_abstained": judged,
        "methods_agree": fuzzy == judged,

        "is_abstained": judged,
    }


In [26]:
classification_rows = []

for i, row in ook_generation_results.iterrows():
    print(f"[{i + 1}/{len(ook_generation_results)}] Classifying {row['question_id']}...")
    result = classify_abstention_row(row["question"], row["answer"])
    classification_rows.append({"question_id": row["question_id"], **result})

ook_classified = ook_generation_results.merge(
    pd.DataFrame(classification_rows), on="question_id", how="left"
)

disagreements = ook_classified[~ook_classified["methods_agree"]]
if len(disagreements):
    print(f"{len(disagreements)} question(s) where the fuzzy check and judge disagreed — worth a manual read:")
    display(disagreements[["question_id", "question", "answer", "fuzzy_abstained", "judge_abstained"]])

display(ook_classified[["question_id", "persona", "subtype", "question", "answer", "is_abstained"]])


[1/18] Classifying C14Q02...
[2/18] Classifying C18Q01...
[3/18] Classifying C27Q01...
[4/18] Classifying C27Q02...
[5/18] Classifying C28Q01...
[6/18] Classifying C28Q02...
[7/18] Classifying C29Q02...
[8/18] Classifying C29Q01...
[9/18] Classifying C30Q01...
[10/18] Classifying C31Q01...
[11/18] Classifying C31Q02...
[12/18] Classifying C32Q01...
[13/18] Classifying C33Q01...
[14/18] Classifying C33Q02...
[15/18] Classifying C34Q01...
[16/18] Classifying C35Q01...
[17/18] Classifying C35Q02...
[18/18] Classifying C52Q01...
1 question(s) where the fuzzy check and judge disagreed — worth a manual read:


,question_id,question,answer,fuzzy_abstained,judge_abstained
14,C34Q01,What job openings exist in the industry for the Bachelor of Games?,"The Bachelor of Games at RMIT has job openings in the industry, particularly through its connections to the local independent games indu...",False,True


,question_id,persona,subtype,question,answer,is_abstained
0,C14Q02,prospective student,out_of_scope_topic,What software do I need to have in order to complete the Bachelor of games?,"To complete the Bachelor of Games, you do not need to have specific software, as the program is more focused on the design and developme...",False
1,C18Q01,current student,kb_gap,Can I transfer between the Bachelor of Business to the Bachelor of Commerce and vice versa?,"You can transfer between the Bachelor of Business and the Bachelor of Commerce, but the transfer is not explicitly stated in the provide...",False
2,C27Q01,prospective student,subjective,Is the Bachelor of Graphic Design harder than the Bachelor of Games?,I don't have enough information in the provided RMIT sources to answer this question.,True
3,C27Q02,prospective student,subjective,"Which is more difficult, the Bachelor of Games or Graphic Design?",I don't have enough information in the provided RMIT sources to answer this question.,True
4,C28Q01,prospective student,predictive,Is the ATAR for Bachelor of Business going to change in 2028?,I don't have enough information in the provided RMIT sources to answer this question.,True
5,C28Q02,prospective student,predictive,What ATAR will I need for the Bachelor of Business in 2028?,I don't have enough information in the provided RMIT sources to answer this question.,True
6,C29Q02,current student,out_of_scope_topic,What is my employability looking like after completing a Bachelor of Games?,I don't have enough information in the provided RMIT sources to answer this question.,True
7,C29Q01,prospective student,subjective,Which degree has the best employment rate?,I don't have enough information in the provided RMIT sources to answer this question.,True
8,C30Q01,current student,subjective,Which lecturer is the best for the Bachelor of Commerce subjects?,I don't have enough information in the provided RMIT sources to answer this question.,True
9,C31Q01,prospective student,personal_circumstance,What are the chances my application will be accepted?,I don't have enough information in the provided RMIT sources to answer this question.,True


## Out-of-KB Abstention Rate

In [27]:
ook_abstention_rate = ook_classified["is_abstained"].mean()

print(f"Out-of-KB Abstention Rate (overall): {ook_abstention_rate:.1%}  "
      f"({ook_classified['is_abstained'].sum()}/{len(ook_classified)} correctly refused)")

ook_by_subtype = (
    ook_classified
    .groupby("subtype")["is_abstained"]
    .agg(["mean", "sum", "count"])
    .rename(columns={"mean": "abstention_rate", "sum": "n_correct_refusals", "count": "n_questions"})
    .round(3)
    .sort_values("abstention_rate")
)
display(ook_by_subtype)

ook_by_persona = (
    ook_classified
    .groupby("persona")["is_abstained"]
    .agg(["mean", "sum", "count"])
    .rename(columns={"mean": "abstention_rate", "sum": "n_correct_refusals", "count": "n_questions"})
    .round(3)
)
display(ook_by_persona)

hallucinations = ook_classified[~ook_classified["is_abstained"]].copy()
print(f"\nHallucinated / attempted answers on out-of-KB questions: {len(hallucinations)}")
display(hallucinations[["question_id", "persona", "subtype", "question", "answer", "retrieved_passage_ids"]])


Out-of-KB Abstention Rate (overall): 83.3%  (15/18 correctly refused)


,abstention_rate,n_correct_refusals,n_questions
subtype,,,
kb_gap,0.000,0,1
predictive,0.800,4,5
out_of_scope_topic,0.833,5,6
personal_circumstance,1.000,2,2
subjective,1.000,4,4


,abstention_rate,n_correct_refusals,n_questions
persona,,,
current student,0.875,7,8
parent/guardian,1.000,1,1
prospective student,0.778,7,9



Hallucinated / attempted answers on out-of-KB questions: 3


,question_id,persona,subtype,question,answer,retrieved_passage_ids
0,C14Q02,prospective student,out_of_scope_topic,What software do I need to have in order to complete the Bachelor of games?,"To complete the Bachelor of Games, you do not need to have specific software, as the program is more focused on the design and developme...","[P11, P18, P32]"
1,C18Q01,current student,kb_gap,Can I transfer between the Bachelor of Business to the Bachelor of Commerce and vice versa?,"You can transfer between the Bachelor of Business and the Bachelor of Commerce, but the transfer is not explicitly stated in the provide...","[P15, P44, P25]"
16,C35Q02,prospective student,predictive,Are there plans for any new majors for the Bachelor of Games?,There are no plans for new majors for the Bachelor of Games.,"[P40, P10, P28]"


## False Abstentions

Of the questions the KB *can* answer, how often does the system
incorrectly refuse? Testing on all 52 `known` questions

In [28]:
known_topics = topics[topics["status"] == "known"].copy()

print("Known questions being tested for false abstention:", len(known_topics))

known_generation_rows = []

for i, row in known_topics.iterrows():
    print(f"[{len(known_generation_rows) + 1}/{len(known_topics)}] Generating {row['question_id']}...")

    result = generate_answer(row["question"])

    known_generation_rows.append({
        "question_id": row["question_id"],
        "persona": row["persona"],
        "question": row["question"],
        "answer": result["answer"],
        "retrieved_passage_ids": result["retrieved_passage_ids"],
        "retrieval_context": result["retrieval_context"],
    })

    pd.DataFrame(known_generation_rows).to_pickle("known_generation_results_progress.pkl")

known_generation_results = pd.DataFrame(known_generation_rows)
print("Known-question generation complete.")


Known questions being tested for false abstention: 52
[1/52] Generating C01Q01...
[2/52] Generating C01Q02...
[3/52] Generating C02Q01...
[4/52] Generating C02Q02...
[5/52] Generating C03Q01...
[6/52] Generating C03Q02...
[7/52] Generating C04Q01...
[8/52] Generating C04Q02...
[9/52] Generating C05Q01...
[10/52] Generating C05Q02...
[11/52] Generating C06Q01...
[12/52] Generating C06Q02...
[13/52] Generating C07Q01...
[14/52] Generating C07Q02...
[15/52] Generating C08Q01...
[16/52] Generating C08Q02...
[17/52] Generating C09Q01...
[18/52] Generating C09Q02...
[19/52] Generating C10Q01...
[20/52] Generating C10Q02...
[21/52] Generating C11Q01...
[22/52] Generating C12Q01...
[23/52] Generating C13Q01...
[24/52] Generating C14Q01...
[25/52] Generating C15Q01...
[26/52] Generating C16Q01...
[27/52] Generating C17Q01...
[28/52] Generating C17Q02...
[29/52] Generating C19Q01...
[30/52] Generating C20Q01...
[31/52] Generating C21Q01...
[32/52] Generating C22Q01...
[33/52] Generating C23Q01..

In [29]:
known_classification_rows = []

for i, row in known_generation_results.iterrows():
    print(f"[{i + 1}/{len(known_generation_results)}] Classifying {row['question_id']}...")
    result = classify_abstention_row(row["question"], row["answer"])
    known_classification_rows.append({"question_id": row["question_id"], **result})

known_classified = known_generation_results.merge(
    pd.DataFrame(known_classification_rows), on="question_id", how="left"
)

false_abstention_rate = known_classified["is_abstained"].mean()

print(f"False Abstention Rate: {false_abstention_rate:.1%}  "
      f"({known_classified['is_abstained'].sum()}/{len(known_classified)} known questions wrongly refused)")

false_abstentions = known_classified[known_classified["is_abstained"]].copy()
display(false_abstentions[["question_id", "persona", "question", "answer", "retrieved_passage_ids"]])


[1/52] Classifying C01Q01...
[2/52] Classifying C01Q02...
[3/52] Classifying C02Q01...
[4/52] Classifying C02Q02...
[5/52] Classifying C03Q01...
[6/52] Classifying C03Q02...
[7/52] Classifying C04Q01...
[8/52] Classifying C04Q02...
[9/52] Classifying C05Q01...
[10/52] Classifying C05Q02...
[11/52] Classifying C06Q01...
[12/52] Classifying C06Q02...
[13/52] Classifying C07Q01...
[14/52] Classifying C07Q02...
[15/52] Classifying C08Q01...
[16/52] Classifying C08Q02...
[17/52] Classifying C09Q01...
[18/52] Classifying C09Q02...
[19/52] Classifying C10Q01...
[20/52] Classifying C10Q02...
[21/52] Classifying C11Q01...
[22/52] Classifying C12Q01...
[23/52] Classifying C13Q01...
[24/52] Classifying C14Q01...
[25/52] Classifying C15Q01...
[26/52] Classifying C16Q01...
[27/52] Classifying C17Q01...
[28/52] Classifying C17Q02...
[29/52] Classifying C19Q01...
[30/52] Classifying C20Q01...
[31/52] Classifying C21Q01...
[32/52] Classifying C22Q01...
[33/52] Classifying C23Q01...
[34/52] Classifying

,question_id,persona,question,answer,retrieved_passage_ids
9,C05Q02,prospective student,What courses does Bachelor of Graphic Design offer?,"The Bachelor of Graphic Design offers the following majors: Experience Design, Branding, Illustration, and Typography.","[P15, P25, P40]"
24,C15Q01,current student,Can I go on to a Master's after completing a Bachelor of Business?,"You can go on to a Master's after completing a Bachelor of Business, as you may be eligible for entry into the Master of Commerce after ...","[P44, P10, P04]"
28,C19Q01,current student,How will work-integrated learning work in my degree?,"Work-integrated learning in your degree will involve completing Work Integrated Learning (WIL) subjects, which are integrated into your ...","[P13, P22, P37]"
29,C20Q01,current student,How can I structure my degree part-time if I plan on working part time?,"To structure your degree part-time while working part-time, you can complete four courses per year over six years, as part-time students...","[P15, P27, P35]"
33,C24Q01,parent/guardian,How much does the Bachelor of Business cost?,The cost of the Bachelor of Business is not specified in the provided information.,"[P14, P32, P44]"
34,C25Q01,parent/guardian,What computer specifications will my child need for the Bachelor of games?,I don't have enough information in the provided RMIT sources to answer this question.,"[P18, P33, P07]"


### Cross-check false abstentions against retrieval quality

In [30]:
false_abstentions_with_retrieval = false_abstentions.merge(
    retrieval_results[["question_id", f"hit@{TOP_K}"]], on="question_id", how="left"
)
display(false_abstentions_with_retrieval[
    ["question_id", "persona", "question", f"hit@{TOP_K}", "answer"]
])

n_generation_fault = (false_abstentions_with_retrieval[f"hit@{TOP_K}"] == 1).sum()
print(f"\nFalse abstentions where retrieval WAS successful (generation-stage fault): {n_generation_fault}")


,question_id,persona,question,hit@3,answer
0,C05Q02,prospective student,What courses does Bachelor of Graphic Design offer?,0,"The Bachelor of Graphic Design offers the following majors: Experience Design, Branding, Illustration, and Typography."
1,C15Q01,current student,Can I go on to a Master's after completing a Bachelor of Business?,1,"You can go on to a Master's after completing a Bachelor of Business, as you may be eligible for entry into the Master of Commerce after ..."
2,C19Q01,current student,How will work-integrated learning work in my degree?,1,"Work-integrated learning in your degree will involve completing Work Integrated Learning (WIL) subjects, which are integrated into your ..."
3,C20Q01,current student,How can I structure my degree part-time if I plan on working part time?,1,"To structure your degree part-time while working part-time, you can complete four courses per year over six years, as part-time students..."
4,C24Q01,parent/guardian,How much does the Bachelor of Business cost?,0,The cost of the Bachelor of Business is not specified in the provided information.
5,C25Q01,parent/guardian,What computer specifications will my child need for the Bachelor of games?,1,I don't have enough information in the provided RMIT sources to answer this question.



False abstentions where retrieval WAS successful (generation-stage fault): 4


## Robustness confusion matrix

In [31]:
confusion_rows = [
    {"kb_status": "known", "model_behaviour": "answered",
     "count": int((~known_classified["is_abstained"]).sum()), "label": "Correct answer"},
    {"kb_status": "known", "model_behaviour": "abstained",
     "count": int(known_classified["is_abstained"].sum()), "label": "False abstention"},
    {"kb_status": "out-of-kb", "model_behaviour": "abstained",
     "count": int(ook_classified["is_abstained"].sum()), "label": "Correct refusal"},
    {"kb_status": "out-of-kb", "model_behaviour": "answered",
     "count": int((~ook_classified["is_abstained"]).sum()), "label": "Hallucination"},
]

confusion_df = pd.DataFrame(confusion_rows)
display(confusion_df)

robustness_summary = pd.DataFrame({
    "metric": [
        "Out-of-KB Abstention Rate (correct refusals)",
        "Hallucination Rate on out-of-KB questions",
        "False Abstention Rate (on known questions)",
        "Overall Robustness Accuracy",
    ],
    "score": [
        ook_classified["is_abstained"].mean(),
        (~ook_classified["is_abstained"]).mean(),
        known_classified["is_abstained"].mean(),
        (
            (~known_classified["is_abstained"]).sum() + ook_classified["is_abstained"].sum()
        ) / (len(known_classified) + len(ook_classified)),
    ],
}).round(3)

display(robustness_summary)


,kb_status,model_behaviour,count,label
0,known,answered,46,Correct answer
1,known,abstained,6,False abstention
2,out-of-kb,abstained,15,Correct refusal
3,out-of-kb,answered,3,Hallucination


,metric,score
0,Out-of-KB Abstention Rate (correct refusals),0.833
1,Hallucination Rate on out-of-KB questions,0.167
2,False Abstention Rate (on known questions),0.115
3,Overall Robustness Accuracy,0.871


## Save robustness results

In [32]:
ook_classified.to_csv("robustness_out_of_kb_results.csv", index=False)
known_classified.to_csv("robustness_false_abstention_results.csv", index=False)
confusion_df.to_csv("robustness_confusion_matrix.csv", index=False)
robustness_summary.to_csv("robustness_summary.csv", index=False)
ook_by_subtype.to_csv("robustness_by_subtype.csv")
ook_by_persona.to_csv("robustness_by_persona.csv")

print("Saved: robustness_out_of_kb_results.csv, robustness_false_abstention_results.csv, "
      "robustness_confusion_matrix.csv, robustness_summary.csv, "
      "robustness_by_subtype.csv, robustness_by_persona.csv")


Saved: robustness_out_of_kb_results.csv, robustness_false_abstention_results.csv, robustness_confusion_matrix.csv, robustness_summary.csv, robustness_by_subtype.csv, robustness_by_persona.csv


## Combine results + summary of preliminary findings

In [33]:
deepeval_results = (
    generation_results
    .merge(relevancy_results, on="question_id", how="left")
    .merge(faithfulness_results, on="question_id", how="left")
    .merge(readability_results, on=["question_id", "persona"], how="left")
)

display(
    deepeval_results[
        [
            "question_id",
            "persona",
            "question",
            "answer",
            "answer_relevancy",
            "faithfulness",
            "flesch_reading_ease",
            "flesch_kincaid_grade",
            "gunning_fog",
            "smog_index"
        ]
    ]
)


NameError: name 'relevancy_results' is not defined

In [34]:
deepeval_summary = pd.DataFrame({
    "metric": ["Answer Relevancy", "Faithfulness"],
    "score": [
        deepeval_results["answer_relevancy"].mean(),
        deepeval_results["faithfulness"].mean()
    ]
})

display(deepeval_summary.round(3))


NameError: name 'deepeval_results' is not defined

In [35]:
deepeval_by_persona = (
    deepeval_results
    .groupby("persona")[["answer_relevancy", "faithfulness"]]
    .mean()
    .round(3)
)

display(deepeval_by_persona)


NameError: name 'deepeval_results' is not defined

In [36]:
weak_cases = deepeval_results[
    (deepeval_results["answer_relevancy"] <= 0.5) |
    (deepeval_results["faithfulness"] <= 0.5)
].copy()

display(
    weak_cases[
        [
            "question_id",
            "persona",
            "question",
            "answer",
            "retrieved_passage_ids",
            "answer_relevancy",
            "faithfulness"
        ]
    ]
)

print("Weak cases:", len(weak_cases))


NameError: name 'deepeval_results' is not defined

In [37]:
export_df = deepeval_results.copy()

export_df["retrieved_passage_ids"] = export_df[
    "retrieved_passage_ids"
].apply(lambda x: " | ".join(map(str, x)))

export_df["retrieval_context"] = export_df[
    "retrieval_context"
].apply(lambda x: " ||| ".join(map(str, x)))

export_df.to_csv("deepeval_results.csv", index=False)
deepeval_summary.to_csv("deepeval_summary.csv", index=False)
deepeval_by_persona.to_csv("deepeval_by_persona.csv")

print("DeepEval results saved.")


NameError: name 'deepeval_results' is not defined

## Overall summary table

One table combining all four evaluation dimensions (retrieval, generation, robustness, readability), for quick reference in the report and slides.

In [ ]:
overall_summary = pd.concat([
    retrieval_summary.assign(section="Retrieval"),
    deepeval_summary.assign(section="Generation (DeepEval)"),
    robustness_summary.assign(section="Robustness"),
], ignore_index=True)[["section", "metric", "score"]]

display(overall_summary)
overall_summary.to_csv("overall_evaluation_summary.csv", index=False)
print("Saved: overall_evaluation_summary.csv")


## Preliminary Evaluation Summary

The preliminary evaluation indicates that the RAG system performs reasonably well overall, while also identifying areas for improvement in retrieval, answer generation, and robustness.

### Retrieval Performance

Retrieval was evaluated across 53 questions using Hit@3, Recall@3, and NDCG@3. The system achieved:

- **Hit@3 = 0.774**
- **Recall@3 = 0.717**
- **NDCG@3 = 0.686**

The Hit@3 score indicates that at least one relevant passage was retrieved within the top three results for approximately 77% of evaluated questions. The lower Recall@3 score suggests that, although the system frequently retrieves relevant information, it does not always retrieve all relevant passages within the top three results. The NDCG@3 score of 0.686 further indicates that relevant passages are not always ranked in the optimal order.

Performance also varied across personas. Current-student questions achieved the strongest retrieval performance (Hit@3 = 0.947, Recall@3 = 0.868, NDCG@3 = 0.850), while parent/guardian and prospective-student questions produced lower scores. Prospective-student questions had the lowest Hit@3 (0.667) and Recall@3 (0.619), suggesting that retrieval could be improved for questions relating to areas such as program requirements, course information, and career outcomes.

### Generation Performance

DeepEval was applied to a stratified sample of 12 questions, with four questions selected from each persona. The system achieved:

- **Answer Relevancy = 0.955**
- **Faithfulness = 0.833**

The high Answer Relevancy score suggests that generated responses generally addressed the user's question directly. Faithfulness was also relatively high, indicating that most answers were supported by the retrieved RMIT passages.

However, individual failures demonstrate why retrieval and generation need to be evaluated separately. For example, for **C42Q01**, the correct passage was retrieved and stated that part-time Bachelor of Business students may complete the degree over six years. Despite this, the generated response incorrectly stated that part-time study takes 36 months. This represents a generation/faithfulness failure despite successful retrieval.

DeepEval also assigned a faithfulness score of 0 to **C25Q01**, where the system abstained because the retrieved sources did not provide sufficient information about required computer specifications. Manual inspection suggests that this abstention was appropriate, highlighting that automated LLM-based evaluation scores should be interpreted alongside qualitative inspection rather than treated as definitive measures of answer quality.

Generation performance also differed by persona. Current-student questions achieved perfect scores for both Answer Relevancy and Faithfulness in this sample, while parent/guardian and prospective-student questions showed weaker faithfulness (0.75 each). Prospective-student questions also had lower Answer Relevancy (0.833).

### Readability

Readability metrics were also calculated for the 12 generated answers. Flesch Reading Ease describes general reading ease, while Flesch-Kincaid Grade Level estimates the educational grade level associated with the text. Average words per sentence and average syllables per word provide additional measures of answer complexity. These metrics describe the linguistic characteristics of the responses and complement the relevance and faithfulness measures.

### Readability Performance

Readability was evaluated on the generated answers using Flesch Reading Ease, Flesch-Kincaid Grade Level, Gunning Fog Index, and SMOG Index, alongside basic answer-length and complexity measures. Flesch Reading Ease provides a measure of reading difficulty where higher scores indicate easier text, while the grade-level measures estimate the education level associated with understanding the response. The Gunning Fog and SMOG measures provide additional estimates based particularly on sentence length and the presence of complex or multi-syllable words.

Because many chatbot answers are relatively short, the SMOG and Gunning Fog results should be interpreted as indicators of linguistic complexity rather than definitive measures of user comprehension. In particular, the standard SMOG formula is designed for longer passages and may be less stable for very short responses. Readability metrics therefore complement, rather than replace, the relevancy and faithfulness measures and qualitative inspection of individual answers.

### Robustness

Robustness was evaluated using all 18 out-of-KB questions and all 52 known questions. The system achieved an Out-of-KB Abstention Rate of 83.3% (15/18 correctly refused) and a False Abstention Rate of 11.5% (6/52 wrongly refused).

Failures were concentrated in predictive questions (e.g. falsely denying any planned new majors) and one knowledge-base gap, C18Q01, where the system fabricated an answer about transferring between Business and Commerce rather than abstaining — unlike the other out-of-KB failures, this is a content gap rather than a robustness limitation. Of the 6 false abstentions, 4 occurred despite successful retrieval, suggesting the generator's abstention threshold is slightly too conservative.

### Overall Interpretation

Overall, the preliminary results suggest that the system is capable of producing highly relevant and generally well-grounded responses when suitable evidence is retrieved. However, retrieval remains the main area for improvement, particularly for prospective-student and parent/guardian questions. The results also demonstrate that successful retrieval does not guarantee a correct generated answer, as the model may select or interpret retrieved evidence incorrectly.

These findings support the use of a multi-layer evaluation framework combining retrieval metrics (Hit@3, Recall@3 and NDCG@3) with generation metrics (Answer Relevancy and Faithfulness) and manual failure analysis. Future improvements should focus on improving retrieval coverage and ranking, reducing conflicting or irrelevant retrieved passages, and strengthening the generation stage so that answers prioritise the most relevant evidence in the retrieved context.

As this is a preliminary evaluation using a limited DeepEval sample of 12 questions, the generation results should be interpreted as indicative rather than as a definitive estimate of system performance.
